In [9]:
#| echo: false

# Ces valeurs servent de secours par défaut si rien n'est passé en ligne de commande
lang = "fr"
id_report = 42

In [10]:
#| label: init-modules
#| echo: false
#| output: asis
import sys
import importlib
from pathlib import Path
import pandas as pd #temporaire pour tester le empty
import json
import os
# from core.db.db_manager import ReportDataManager
# from core.pdf import pdf_components as comp
# from core.typst_utils import build_section_title, build_txt_block, build_table_block 
# # Force le rechargement si les fichiers .py ont changé sur le disque
# importlib.reload(core.db.db_manager)
# importlib.reload(core.pdf.pdf_components)
# 1. Imports initiaux complets nécessaires pour importlib.reload
import core.db.db_manager
import core.pdf.pdf_components

# 2. Forcer le rechargement propre depuis le disque
importlib.reload(core.db.db_manager)
importlib.reload(core.pdf.pdf_components)

# 3. Extraction de vos alias et classes comme dans votre code d'origine
from core.db.db_manager import ReportDataManager
from core.pdf import pdf_components as comp
from core.typst_utils import build_section_title, build_txt_block, build_table_block 

# --- RECRUTEMENT SÉCURISÉ DES PARAMÈTRES ---

# --- RÉCUPÉRATION DIRECTE ET SÉCURISÉE DES VARIABLES ---
# Quarto a automatiquement écrasé 'lang' et 'id_report' si configuré dans le CLI ou le YAML
lang_param = lang
id_report_param = id_report

# Récupération des paramètres
#lang_param = os.getenv("QUARTO_PARAM_lang", os.getenv("QUARTO_PARAM_LANG", "fr"))
id_report_param = os.getenv("QUARTO_PARAM_id_report", os.getenv("QUARTO_PARAM_ID_REPORT", 42))
#lang_param = params.get('lang', 'fr')
db = ReportDataManager(lang=lang_param, id_report=id_report_param)
# Récupération optionnelle du chemin de la bannière si elle vient de la DB, 
# ou utilisation d'une valeur par défaut
banner_path = "SPF_BI-LANGUE.jpg" 

# Récupération des textes traduits
txt_gauche = db.get_txt("foot") + "\n" + db.get_txt("foot1")

# On exporte les textes dans un fichier que Quarto/Typst peut lire
import json
with open("core/typst/vars.json", "w", encoding="utf-8") as f:
    json.dump({"footer-left": txt_gauche,
        "banner-path": banner_path}, f)

In [11]:
#| echo: false
#| output: asis
titre = db.get_txt("Z")
print(build_section_title(titre, underline=True))
# print("```{=typst}")
# print("#align(center)[")
# print(f"== {titre}")
# print("]")
# print("#v(10pt)")
# # print("#align(center)[#line(length: 40%, stroke: 1pt + luma(180))]")
# print("#v(15pt)")
# print("```")

```{=typst}
#show heading.where(level: 2): it => [
  #set align(center)
  #set text(weight: "bold")
  #it.body
]
== RHM
#v(10pt)
#align(center)[#line(length: 40%, stroke: 1pt + luma(180))]
#v(15pt)
```


In [12]:
#| echo: false
#| output: asis
df_hosp = db.get_hospital_data()
table_md = comp.generate_hospital_table(df_hosp, db)
print(build_table_block(table_md))
# print("```{=typst}")
# print("#align(center)[")
# print(table_md)
# print("]")
# print("```")

```{=typst}
#set text(size: 10pt)
#set text(font: "Open Sans")
#align(center)[
#align(center)[
  #table(
    columns: (2fr, 2fr),
    column-gutter: 5pt,
    stroke: white,
    inset: (x: 4pt, y: 2pt),
    [#align(right)[Nom de l'hôpital]] , [#align(left)[CHR Namur]] , [#align(right)[Année]] , [#align(left)[2026]] , [#align(right)[Période]] , [#align(left)[Semestre 1]] , [#align(right)[Date]] , [#align(left)[17/06/2026]]
  )
]
]
```


In [13]:
#| echo: false
#| output: asis
titre = db.get_txt("ReportTitle")
print(build_section_title(titre, underline=True))
# print("```{=typst}")
# print("#align(center)[")
# print(f"== {titre}")
# print("]")
# print("#v(15pt)")
# print("```")

```{=typst}
#show heading.where(level: 2): it => [
  #set align(center)
  #set text(weight: "bold")
  #it.body
]
== Nombre d'erreurs pour chaque contrôle
#v(10pt)
#align(center)[#line(length: 40%, stroke: 1pt + luma(180))]
#v(15pt)
```


In [14]:
#| label: tableau-erreurs
#| warning: false
#| echo: false
#| output: true
df_err = db.get_errors_data()
comp.generate_severity_errors_table(df_err, db)

Sévérité,Contrôle,Description,Nombre
1,ERR-101,Panne serveur,5
1,ERR-102,Crash DB,2
Total 1,,,7
2,ERR-201,Timeout API,12
2,ERR-202,Disque 80%,4
Total 2,,,16
3,ERR-301,Session exp.,45
3,ERR-302,Lenteur,8
Total 3,,,53
4,ERR-401,Mauvaise performance,3


In [15]:
#| echo: false
#| output: asis
#txt = f"{db.get_txt('sev11')}{db.get_txt('sev12')}{db.get_txt('sev21')}{db.get_txt('sev22')}{db.get_txt('sev31')}{db.get_txt('sev32')}{db.get_txt('sev41')}{db.get_txt('sev42')}"
txt = "\n".join([
    db.get_txt('sev11'),
    db.get_txt('sev12'),
    db.get_txt('sev21'),
    db.get_txt('sev22'),
    db.get_txt('sev31'),
    db.get_txt('sev32'),
    db.get_txt('sev41'),
    db.get_txt('sev42')
])
print(build_txt_block(txt))
# print("```{=typst}")
# print("#v(25pt)")
# print("#par(justify: true)[")
# print(txt)
# print("]")
# print("```")

```{=typst}
#set text(size: 8pt)
#set text(font: "Open Sans")
#set text(fill: black)
#align(left)[
Severity 1 = Erreur bloquante. #linebreak() Les erreurs bloquantes sont des erreurs fatales et doivent absolument être corrigées avant de pouvoir aller plus loin. #linebreak() Severity 2 = Erreur non bloquante. #linebreak() Les erreurs non bloquantes doivent être vérifiées et contrôlées par rapport à la réalité. Si les données reflètent la situation réelle, elles ne doivent pas être modifiées. Dans le cas contraire, la modification est obligatoire. #linebreak() Severity 3 = Avertissement. #linebreak() Les erreurs de sévérité 3 attirent l'attention sur des valeurs de champ qui pourraient être incorrectes. Une correction des données n'est pas nécessairement obligatoire, puisqu'il s'agit d'avertissements. #linebreak() Severity 4 = Erreur bloquante future #linebreak() Les erreurs de sévérité 4 deviendront des erreurs bloquantes dans le futur.
]
```


```{=typst}
#pagebreak()
```

In [16]:
#| echo: false
#| output: asis
titre = db.get_txt("ReportTitle")
print(build_section_title(titre, underline=True))

```{=typst}
#show heading.where(level: 2): it => [
  #set align(center)
  #set text(weight: "bold")
  #it.body
]
== Nombre d'erreurs pour chaque contrôle
#v(10pt)
#align(center)[#line(length: 40%, stroke: 1pt + luma(180))]
#v(15pt)
```


In [17]:
#| label: tableau-erreurs2
#| warning: false
#| echo: false
#| output: asis
# table_err = comp.generate_severity_errors_table_typst(df_err, db)
# 1. On vérifie si le DataFrame contient des données
# décommenter ligne ci-dessous pour tester le cas où il est vide
df_err = pd.DataFrame()
#if df_err.empty:
    # On utilise votre fonction existante pour afficher la note en petite taille (8pt)
    # txt=db.get_txt('labelerr')
    # msg_pas_derreur = build_txt_block(
    #     txt, 
    #     size="12pt", 
    #     bold=True, 
    #     color="gray"
    # )
    # print(msg_pas_derreur)
if df_err.empty:
    # On écrit directement le bloc Typst brut au format textuel sans fonction intermédiaire
    congratulations_txt = db.get_txt('labelcongrats')
    conclusion = db.get_txt('labelerr') 
    
    print("```{=typst}")
    print("#v(10pt)")
    print("#rect(")
    print("  width: 100%,")
    print("  fill: rgb(\"#e6f4ea\"),")
    print("  stroke: 1pt + rgb(\"#137333\"),")
    print("  radius: 4pt,")
    print("  inset: 15pt")
    print(")[")
    print("  #grid(")
    print("    columns: (auto, 1fr),")
    print("    gutter: 12pt,")
    print("    align: (center + horizon, left + horizon),")
    print("    text(size: 20pt)[✅],")
    print("    [")
    # Utilisation de f"..." pour injecter dynamiquement des variables de la DB
    print(f"      #text(weight: \"bold\", size: 11pt, fill: rgb(\"#137333\"))[{congratulations_txt}] \\")
    print(f"      #text(size: 10pt, fill: rgb(\"#202124\"))[{conclusion}]")
    print("    ]")
    print("  )")
    print("]")
    print("#v(10pt)")
    print("```")


else:
    # 2. Si le DataFrame n'est pas vide, on génère et on affiche le tableau normalement
    table_err = comp.generate_severity_errors_table_typst4(df_err, db)
    print("```{=typst}")
    print(table_err)
    print("```")
    txt = "\n".join([
    db.get_txt('sev11'),
    db.get_txt('sev12'),
    db.get_txt('sev21'),
    db.get_txt('sev22'),
    db.get_txt('sev31'),
    db.get_txt('sev32'),
    db.get_txt('sev41'),
    db.get_txt('sev42')
    ])
    print(build_txt_block(txt))

```{=typst}
#v(10pt)
#rect(
  width: 100%,
  fill: rgb("#e6f4ea"),
  stroke: 1pt + rgb("#137333"),
  radius: 4pt,
  inset: 15pt
)[
  #grid(
    columns: (auto, 1fr),
    gutter: 12pt,
    align: (center + horizon, left + horizon),
    text(size: 20pt)[✅],
    [
      #text(weight: "bold", size: 11pt, fill: rgb("#137333"))[ Félicitations] \
      #text(size: 10pt, fill: rgb("#202124"))[Il n'y a pas d'erreurs]
    ]
  )
]
#v(10pt)
```


```{=typst}
#page_landscape()
#title_block("TEST_SECTION 0")
```

In [18]:
#| echo: false
#| output: asis
titre = db.get_txt("ReportTitle")
print("```{=typst}")
print(f"#title_block({json.dumps(titre, ensure_ascii=False)})")
print("```")

```{=typst}
#title_block("Nombre d'erreurs pour chaque contrôle")
```


In [19]:
#| echo: false
#| output: asis
titre = db.get_txt("ReportTitle")
print("```{=typst}")
print(f"#section_landscape([{f'#title_block({json.dumps(titre, ensure_ascii=False)})'}])")
print("```")

df_hosp = db.get_hospital_data()
table_md = comp.generate_hospital_table(df_hosp, db)
print("```{=typst}")
print("#align(center)[")
print(table_md)
print("]")
print("```")

```{=typst}
#section_landscape([#title_block("Nombre d'erreurs pour chaque contrôle")])
```
```{=typst}
#align(center)[
#align(center)[
  #table(
    columns: (2fr, 2fr),
    column-gutter: 5pt,
    stroke: white,
    inset: (x: 4pt, y: 2pt),
    [#align(right)[Nom de l'hôpital]] , [#align(left)[CHR Namur]] , [#align(right)[Année]] , [#align(left)[2026]] , [#align(right)[Période]] , [#align(left)[Semestre 1]] , [#align(right)[Date]] , [#align(left)[17/06/2026]]
  )
]
]
```


In [20]:
#| echo: false
#| output: asis



# 1. Récupération des données brutes
raw_data = db.get_admission_data()

# 2. Récupération du code natif Typst
code_typst = comp.generate_admission_table_typst(raw_data,db)

# 3. Injection directe dans le rendu de la page
print("```{=typst}")
print("#align(center)[")
print(code_typst)
print("]")
print("```")

```{=typst}
#align(center)[
#set text(size: 8pt)
#v(8pt)
#table(
  columns: (2.5fr, 1fr, 1fr, 1fr, 1fr, 1fr, 1fr, 1fr, 1fr, 1fr, 1fr, 1fr, 1fr),
  align: (left, right, right, right, right, right, right, right, right, right, right, right, right),
  stroke: 0.3pt + rgb("#e0e0e0"),
  fill: (x, y) => 
    if y == 0 or y == 1 { rgb("#1a365d") }
    else if y == 12 { rgb("#edf2f7") }
    else if y >= 13 { rgb("#f8fafc") }
    else if calc.even(y) { rgb("#f8fafc") }
    else { none },
  table.cell(rowspan: 2, align: center + horizon, stroke: none)[#set text(fill: white); *Code & Description Diagnostic*],
  table.cell(colspan: 3, align: center, stroke: none)[#set text(fill: white); *Hospitalisation (H)*],
  table.cell(colspan: 3, align: center, stroke: none)[#set text(fill: white); *Ambulatoire (F)*],
  table.cell(colspan: 3, align: center, stroke: none)[#set text(fill: white); *Médecine (M)*],
  table.cell(colspan: 3, align: center, stroke: none)[#set text(fill: white); *Long Séjour (L)*],
  